In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/FC26_20250921.csv")
df.drop(columns=["fifa_version", "fifa_update", "fifa_update_date", "player_face_url", "work_rate", "player_url"], inplace=True)

df_draft = df[df['overall'] >= 75].copy()

faixas = [
        (df_draft['overall'] >= 87),
        (df_draft['overall'] >= 84) & (df_draft['overall'] < 87),
        (df_draft['overall'] >= 80) & (df_draft['overall'] < 84),
        (df_draft['overall'] < 80)
]
    
pesos = [5, 15, 40, 80]
df_draft['weight'] = np.select(faixas, pesos, default=0)

/tmp/ipykernel_21994/387547489.py:4: DtypeWarning: Columns (0: player_tags) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/FC26_20250921.csv")


In [32]:
from draft import generate_draft_pack
from bot import Bot
from eafc_utils import f

print("===========================================\n INICIANDO SIMULADOR DE DRAFT EA FC \n===========================================")

# 4-3-3
formacao_433 = ['ST', 'LW', 'RW', 'CM', 'CM', 'CM', 'LB', 'CB', 'CB', 'RB', 'GK']
remaining_positions = list(formacao_433) # Deep copy 
meu_time = []
ids_escolhidos = set()

NUMERO_DE_ROLLOUTS = 1000

meu_bot = Bot(mode="expectimax", num_rollouts=NUMERO_DE_ROLLOUTS)

for rodada in range(1, 12):
    print(f"\n--- RODADA {rodada}/11 ---")
    pacote = generate_draft_pack(df_draft, rodada, remaining_positions, ids_escolhidos)
    opcoes = pacote.reset_index(drop=True)
    print(opcoes[['short_name', 'overall', 'club_name', 'league_name', 'nationality_name']])

    df_meu_time = pd.DataFrame(meu_time) if meu_time else pd.DataFrame(columns=df_draft.columns)

    indice_escolhido, posicao_escolhida = meu_bot.choose(
        options=opcoes,
        current_squad=df_meu_time,
        db=df_draft,
        current_round=rodada,
        remaining_positions=remaining_positions
    )

    carta_escolhida = opcoes.iloc[indice_escolhido].copy()
    carta_escolhida['choosen_position'] = posicao_escolhida
    remaining_positions.remove(posicao_escolhida)
    meu_time.append(carta_escolhida)
    ids_escolhidos.add(carta_escolhida['player_id'])

    print(f">>> ESCOLHA DO BOT: {carta_escolhida['short_name']} (OVR {carta_escolhida['overall']})")

print("\n===========================================")
print(" DRAFT CONCLUÍDO! SEU ELENCO FINAL: ")
print("===========================================")
df_final = pd.DataFrame(meu_time)
print(df_final[['short_name', 'overall', 'league_name', 'nationality_name', 'club_name']])
nota_final = int(f(df_final))
print(f"\nNOTA FINAL DO TIME:", nota_final)

 INICIANDO SIMULADOR DE DRAFT EA FC 

--- RODADA 1/11 ---
       short_name  overall          club_name     league_name nationality_name
0  R. Lewandowski       88       FC Barcelona         La Liga           Poland
1         Alisson       89          Liverpool  Premier League           Brazil
2           Rodri       90    Manchester City  Premier League            Spain
3         A. Isak       88          Liverpool  Premier League           Sweden
4         H. Kane       89  FC Bayern München      Bundesliga          England
>>> ESCOLHA DO BOT: Rodri (OVR 90)

--- RODADA 2/11 ---
   short_name  overall             club_name          league_name  \
0    M. Smets       76              KRC Genk           Pro League   
1  A. Sánchez       76            Sevilla FC              La Liga   
2  A. Zorgane       78  Union Saint-Gilloise           Pro League   
3    D. Rossi       75         Columbus Crew  Major League Soccer   
4   V. Rosier       77            CA Osasuna              La Liga  